# Deploy a Model

> **Not using MaaS?** If you are deploying a standalone model (without Models-as-a-Service gateway),
> see the [RHOAI Model Deployment Workshop](https://github.com/gymnatics/Red-Hat-Inference-Workshop/blob/main/RHOAI-MODEL-DEPLOYMENT-WORKSHOP.md)
> for a step-by-step guide covering single-model serving with vLLM on OpenShift AI.

Deploy a model via the **RHOAI Dashboard** (Models as a Service), then verify and save the config.

**Prerequisites:**
- `1_environment_setup.ipynb` completed (cluster verified, `.env` configured)
- MaaS infrastructure installed via [RHOAI-Toolkit](https://github.com/hyogrin/RHOAI-Toolkit)

## 1. Deploy Model via RHOAI Dashboard

### Recommended Models

| Model | VRAM | Source (OCI ModelCar) |
|-------|------|----------------------|
| Qwen3.5-35B-A3B MoE FP8 | ~21 GB | `oci://registry.redhat.io/rhai/modelcar-qwen3-5-35b-a3b-fp8-dynamic:3.0` |
| Qwen3-14B | ~14 GB | `oci://quay.io/redhat-ai-services/modelcar-catalog:qwen3-14b` |
| Qwen3-8B | ~8 GB | `oci://quay.io/redhat-ai-services/modelcar-catalog:qwen3-8b` |
| Qwen3-4B | ~4 GB | `oci://quay.io/redhat-ai-services/modelcar-catalog:qwen3-4b` |
| Qwen2.5-7B-Instruct | ~7 GB | `oci://quay.io/redhat-ai-services/modelcar-catalog:qwen2.5-7b-instruct` |

### Custom Parameters (vLLM)

Add these in the Dashboard's **Custom parameters** section as environment variables:

| Key | Value | Why |
|-----|-------|-----|
| `VLLM_ADDITIONAL_ARGS` | See below | vLLM CLI flags |

**Recommended `VLLM_ADDITIONAL_ARGS` value:**

```
--max-model-len=16384
--enforce-eager
--gpu-memory-utilization=0.90
--enable-auto-tool-choice
--tool-call-parser=hermes
--reasoning-parser=qwen3


```

| Flag                                                     | Effect                                                                                                                                                                                                                                                                 |
| -------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `--max-model-len=16384`                                  | Caps the **combined prompt and output length to 16K tokens**. Reducing it from the model’s native context length lowers KV-cache memory requirements and helps prevent OOM.                                                                                            |
| `--enforce-eager`                                        | Disables CUDA graphs and always uses PyTorch eager execution. This can reduce CUDA-graph capture memory and startup overhead, but may lower inference throughput. It is **not always required** for large models.                                                      |
| `--gpu-memory-utilization=0.95`                          | Allows the vLLM model executor to use up to **95% of each GPU’s VRAM**. Remaining memory after model weights and runtime allocations is generally used for KV cache. Default is `0.9`.                                                                                 |
| `--enable-auto-tool-choice`                              | Enables the model to automatically decide whether to return a normal response or invoke one or more tools.                                                                                                                                                             |
| `--tool-call-parser=qwen3_coder`                         | Parses the **Qwen3-Coder-specific tool-call format**, used by models such as `Qwen3-Coder-30B-A3B-Instruct` and `Qwen3-Coder-480B-A35B-Instruct`.                                                                                                                      |
| `--tool-call-parser=hermes`                              | Parses the **Hermes-style `<tool_call>` format**, used by Qwen2.5, QwQ and Hermes-compatible models.                                                                                                                                                                   |
| `--trust-remote-code`                                    | Allows Hugging Face model repositories to load and execute their custom Python model or tokenizer code. Use only with trusted model repositories.                                                                                                                      |
| `--reasoning-parser=qwen3`                               | Separates Qwen3 reasoning output from the final answer and returns it in the OpenAI-compatible `reasoning_content` field. you can turn off to create </think> tag for the coding assistant use case                                                                                                                                              |
| `--chat-template=/etc/chat-template/chat_template.jinja` | Uses the specified Jinja2 chat template to convert OpenAI-format messages, roles and tool definitions into the model-specific prompt format. It is useful when the tokenizer does not provide the correct template or **when a custom tool-calling template is required.** |



> **Without `--max-model-len` and `--enforce-eager`**, large models (27B+) will likely fail to start on a single GPU due to OOM during KV cache allocation.

### Controlling Thinking Mode per Request (Qwen3)

Independently of the vLLM `--reasoning-parser` flag, Qwen3 models support **per-request** control over whether the model generates `<think>` reasoning at all.

| Method | How | Effect |
|--------|-----|--------|
| `chat_template_kwargs` | Pass `{"enable_thinking": false}` in the request body | Disables thinking entirely — no `<think>` block generated, faster response |
| `/no_think` token | Append `/no_think` to the user message | Same effect, controlled inline |
| `/think` token | Append `/think` to the user message | Explicitly enables thinking (default behavior) |

**Example (OpenAI-compatible API):**

```python
response = client.chat.completions.create(
    model="qwen3-8b",
    messages=[{"role": "user", "content": "Hello"}],
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
```

**Summary:**
- `--reasoning-parser=qwen3` (server-level) — model still thinks, but output is **parsed** into a separate `reasoning_content` field
- `enable_thinking: false` (request-level) — model **skips thinking entirely**, reducing latency and token usage

### Open **RHOAI Dashboard** → Create your project 

A "project" (also known as a Kubernetes namespace) is your workspace where you'll deploy models, create workbenches, and run experiments.

![create project](../images/create_project.png)

### Deploy a Model  

In this section, you'll deploy the Qwen3.5-35B model using the Distributed inference with llm-d.

#### Create GPU hardware profiles 

A **hardware profile** defines the compute resources (CPU, memory, GPU) allocated to a model deployment. It also includes tolerations that allow pods to schedule on GPU-tainted nodes.

- Navigate to **Settings > Hardware profiles** in the RHOAI dashboard
- The pre-created `gpu-profile` that includes:
  - GPU resource limits (`nvidia.com/gpu`)
  - Node selector for GPU nodes (`nvidia.com/gpu.present: "true"`)
  - Tolerations for GPU node taints

![create_gpu_profile](../images/create_gpu_profile.png)

#### Start the Model Deployment

Click "Projects" in the left sidebar
Click on your project name (demo)
Click the "Deployments" tab
Click the "Deploy model" button

![deploy_model](../images/deploy_model.png)

Fill in the following:

| Field | What to Select/Enter |
|-------|---------------------|
| **Model location** | `URI` |
| **URI** | `oci://registry.redhat.io/rhai/modelcar-qwen3-5-35b-a3b-fp8-dynamic:3.0` |
| **Name** | `qwen3.5-35b` |
| **Model type** | `Generative AI model (Example, LLM)` |

#### Input the Model Deployment Specs

Configure the deployment settings:

| Field | What to Select/Enter |
|-------|---------------------|
| **Model deployment name** | `qwen3.5-35b` |
| **Hardware profile** | Select your GPU profile |
| **Deployment resource** | `Automatic selection` or `Manual selection` |
| **Number of replicas** | `1` |

![deploy_step](../images/deploy_step1.png)

#### Input the Model Deployment Specs

![deploy_step](../images/deploy_step3.png)

After checking **"Add custom runtime arguments"**, add the following arguments. Enter each on its own line in the text box:

```
--max-model-len=16384
--enforce-eager
--gpu-memory-utilization=0.90
--enable-auto-tool-choice
--tool-call-parser=hermes
--reasoning-parser=qwen3
```

* Different model families require different tool call parsers

Click **"Next"** to proceed.

#### Review the configuration summary. Verify the key settings:

![deploy_step](../images/deploy_confirm.png)

- **Model location:** URI
- **Location details:** `oci://registry.redhat.io/rhai/modelcar-qwen3-5-35b-a3b-fp8-dynamic:3.0`
- **Hardware profile:** gpu-profile
- **Deployment resource:** Distributed inference with llm-d
- **Replicas:** 1
- **AI asset endpoint:** Yes
- **Token authentication:** No
- **Additional runtime arguments**: 6
--max-model-len=16384, --enforce-eager, --gpu-memory-utilization=0.90, --enable-auto-tool-choice, --tool-call-parser=hermes, --reasoning-parser=qwen3
- **MaaS endpoint** Yes

#### Monitor Deployment util Ready

![deploy_step](../images/starting_model.png)
After clicking Deploy, you can monitor the status until Ready

## 2. Get the MaaS API Key

Once the model deployment reaches **Ready** status, you need to create an API key for authenticating requests through the MaaS Gateway.

#### Create a new API Key

Navigate to **AI Assets → API Keys** in the RHOAI Dashboard and click **"Create API key"**. Give it a descriptive name (e.g., `lab-key`) and select the project where your model is deployed.

![create_api_key](../images/create_api_key.png)

#### Copy the API Key

After creation, the key is displayed **only once**. Click the copy icon to save it to your clipboard immediately — you will not be able to view it again.

![copy_api_key](../images/copy_api_key.png)

#### Verify the API Key

The key now appears in the API Keys list with its status and associated project. Store this value in your `.env` file as `MAAS_API_KEY` (the next code cell will handle this automatically).

![api_key_result](../images/api_key_result.png)

## 3. Save Model Config to `.env`

Copy the model endpoint URL and API key from the Dashboard into your local `.env` file. This file is used by all subsequent notebooks to authenticate and route requests to the correct model.

The key variables to set are:
- **`MODEL_NAME`** — the deployed model name (e.g., `qwen3.5-35b`)
- **`MODEL_ENDPOINT`** — the inference URL shown in the Dashboard
- **`MAAS_API_KEY`** — the API key you copied in the previous step

![env_example](../images/env_example.png)

> The code cell below auto-detects these values from the cluster and writes them to `.env`. You can also edit the file manually if needed.

## 4. Test Model Endpoint

Quick inference test via `port-forward` to bypass gateway ext_proc.

In [5]:
import subprocess, json, time, os
from dotenv import load_dotenv

load_dotenv("../.env", override=True)
MODEL_NAME = os.getenv("MODEL_NAME")
NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")

if not MODEL_NAME:
    print("⚠️  MODEL_NAME not set. Run Step 3 first.")
else:
    svc_name = f"{MODEL_NAME}-kserve-workload-svc"
    local_port = 18000

    pf = subprocess.Popen(
        ["oc", "port-forward", f"svc/{svc_name}", f"{local_port}:8000", "-n", NAMESPACE],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    # Wait for port-forward to be ready
    import select
    for _ in range(10):
        time.sleep(1)
        if pf.poll() is not None:
            break
        ready, _, _ = select.select([pf.stdout], [], [], 0)
        if ready:
            line = pf.stdout.readline().decode()
            if "Forwarding" in line:
                break

    if pf.poll() is not None:
        err = pf.stderr.read().decode()
        print(f"❌ port-forward failed: {err}")
    else:
        try:
            r = subprocess.run(
                ["curl", "-sk", "--max-time", "60",
                 f"https://localhost:{local_port}/v1/chat/completions",
                 "-H", "Content-Type: application/json",
                 "-d", json.dumps({
                     "model": MODEL_NAME,
                     "messages": [{"role": "user", "content": "Say hello in one sentence. /no_think"}],
                     "max_tokens": 30,
                     "chat_template_kwargs": {"enable_thinking": false}
                 })],
                capture_output=True, text=True, timeout=65
            )
            if r.returncode == 0 and r.stdout.strip():
                resp = json.loads(r.stdout)
                if "choices" in resp:
                    print(f"✅ Model responds! (via port-forward to {svc_name})")
                    print(f"   {resp['choices'][0]['message']['content']}")
                elif "error" in resp:
                    print(f"❌ API error: {resp['error']}")
                else:
                    print(f"❌ Unexpected: {r.stdout[:300]}")
            else:
                print(f"❌ curl exit={r.returncode}")
                if r.stderr:
                    print(f"   stderr: {r.stderr[:300]}")
                print(f"   Check: oc get pods -n {NAMESPACE} | grep {MODEL_NAME}")
        finally:
            pf.terminate()
            pf.wait()

❌ port-forward failed: Error from server (NotFound): services "redhataiqwen35-35b-a3b-fp8-dyn-kserve-workload-svc" not found



ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hyochoi/dev/rhoai-coding-assistant-lab/.venv/lib/python3.14/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/Users/hyochoi/dev/rhoai-coding-assistant-lab/.venv/lib/python3.14/site-packages/ipykernel/kernelbase.py", line 584, in shell_channel_thread_main
    _, msg2 = self.session.feed_identities(msg, copy=False)
              ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/hyochoi/dev/rhoai-coding-assistant-lab/.venv/lib/python3.14/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hyochoi/dev/rhoai-coding-assistant-lab/.venv/lib/python3.14/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.resu

## Next Steps

- `3_app_setup.ipynb` — Build and deploy the cafe-order-system demo app
- `../1_mcp_servers/2_deploy_mcp_servers.ipynb` — Verify MCP tool servers
- `../2_maas/2_enable_maas.ipynb` — Register models and create API keys